# Data Aggregation

**General Notebook Explanation**

This notebook is responsible for enriching and aggregating the raw trips dataset by integrating multiple data sources into a unified DataFrame.

The process merges external and derived attributes with the raw trip data, including:

- Rainfall data from the Israeli Meteorological Service (IMS)
- Spatial attributes related to public transport paths (“Natatz”)
- Geometrical features of routes derived from GTFS data
- Passenger-related attributes

The result is a comprehensive, feature-rich dataset that combines operational, environmental, and spatial information, and is ready for further analysis and modeling.

### Imports

In [5]:
import pandas as pd
import geopandas as gpd
import zipfile
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))
from src.config import RAIN_COLS_HEB_TO_ENG
from src.data_agg import modify_rainfall_df_values, group_rainfall_by_day_and_hour, merge_rain_to_trips_df
from src.data_agg import route_to_shape_dict, get_linestring_for_shape, add_linestrings_to_route_dict,extract_linestring
from src.data_agg import buffer_and_dissolve_routes, calc_length_within_buffer, calc_curvity, parse_mixed_date, filter_by_multiple_date_windows, add_circular_flag

In [6]:
### Import Trips interim dataframe
df_trips = pd.read_csv(r'../data/interim_data/telaviv_buses_april_2024_cleaned.csv')

## 1. Rainfall
Rainfall data was obtained from the Israeli Meteorological Service (IMS) open data portal:
https://ims.gov.il/he/data_gov

In [7]:
### Importing rainfall data collected by the Israeli Meteorological Service (IMS)
df_rain = pd.read_csv(r'../data/external/rainfall_tlv_apr_2024.csv')

### Renaming its columns to english for easy processing
df_rain = df_rain.rename(columns=RAIN_COLS_HEB_TO_ENG)

### Modifying value types
df_rain = modify_rainfall_df_values(df_rain)

### Grouping (sum) rainfall by day and hour 
rain_grouped_df = group_rainfall_by_day_and_hour(df_rain)

### Merge the rainfall (mm) to th trips df - so each trip will have the amount of rainfall at the time (hour) of the trip
df_trips = merge_rain_to_trips_df(rain_grouped_df,df_trips)

## 2. Spatial attributes

### 2.1. Spatial data imports

In [8]:
#### public transport paths ("Natatz")
public_transport_paths = gpd.read_file("../data/external/shapefiles/Nataz_shape_202601.zip")


#### gtfs data
zip_path_dec24 = '../data/external/gtfs/gtfs.zip'
with zipfile.ZipFile(zip_path_dec24) as z:
    print(z.namelist())  # see file names inside

    df_trips_gtfs_dec24 = pd.read_csv(z.open('trips.txt'))
    df_shapes_gtfs_dec24 = pd.read_csv(z.open('shapes.txt'))
    df_routes_gtfs_dec24 = pd.read_csv(z.open('routes.txt'))

#### gtfs data
zip_path_feb23 = '../data/external/gtfs/gtfs_feb23.zip'
with zipfile.ZipFile(zip_path_feb23) as z:
    print(z.namelist())  # see file names inside

    df_trips_gtfs_feb23 = pd.read_csv(z.open('trips.txt'))
    df_shapes_gtfs_feb23 = pd.read_csv(z.open('shapes.txt'))
    df_routes_gtfs_feb23 = pd.read_csv(z.open('routes.txt'))

['routes.txt', 'shapes.txt', 'trips.txt']
['trips.txt', 'routes.txt', 'shapes.txt']


### 2.2. Process

- Assign to each trips its geometry
- Convert  the trips df to GDF (GeoDataFrame) based on the route geometry (linestring)

In [9]:
### Create a dictionary to each route_id the associated shape_id in gtfs {'route_id_1': {'shape_id': shape_id_1},...}                                                                                    
route_shape_map  = route_to_shape_dict(df_trips_gtfs_dec24, df_trips.route_id.unique())

### Add to the route_shape_map to each route_id the linestinrg extracted from shape_id df {'shape_id': shape_id_1, 'linestring': 'linestring_1},...} 
route_shape_line_map = add_linestrings_to_route_dict(df_shapes_gtfs_dec24, route_shape_map)

### reducing to trips that didn't match with any route linestring
missing_trips = df_trips[~(df_trips['route_id'].isin(route_shape_map.keys()))].copy()

### Repeating the process with older grfs version (from Feb-23)
route_shape_map_feb23  = route_to_shape_dict(df_trips_gtfs_feb23, missing_trips.route_id.unique())
route_shape_line_map_feb23 = add_linestrings_to_route_dict(df_shapes_gtfs_feb23, route_shape_map_feb23)

### Merging the two dictionaries
route_shape_line_map = route_shape_line_map | route_shape_line_map_feb23

### Add this linestring to the trips df as an attribute - "linestring"
df_trips["linestring"] = df_trips["route_id"].map(lambda x: extract_linestring(x, route_shape_line_map))

### Convert the df to GeoDataFrame, using the linestring as the geometry reference
gdf_trips = gpd.GeoDataFrame(df_trips, geometry="linestring", crs="EPSG:4326")

### Handle CRS - make it in ITM for better distance measuring
gdf_trips = gdf_trips.set_crs(epsg=4326)   # only if not already set
gdf_trips = gdf_trips.to_crs(epsg=2039)

- Handle The Public Transport Paths data
- Filter paths that were removed before, or added after the investigated timeframe

In [10]:
public_transport_routes = public_transport_paths.set_crs(epsg=2039, allow_override=True)

cols = ["STARTDATE", "ENDDATE"]
public_transport_paths[cols] = public_transport_paths[cols].apply(lambda col: col.apply(parse_mixed_date))

public_transport_routes_filtered= filter_by_multiple_date_windows(
                                                                public_transport_paths,
                                                                start_cols=["STARTDATE"],
                                                                end_cols=["ENDDATE"],
                                                                start_cutoffs=["2024-01-01"],
                                                                end_cutoffs=["2024-02-01"]
)

- Adding spatial attributes
- Calculating the length within public transport path (based on buffer intersection with it)
- Adding the percentage of the route within this path
- Adding curvity metric (route_length/aerial distance between start and end stops)

In [11]:
### Making 15 m buffer around each public transport path
pt_routes_area_gdf = buffer_and_dissolve_routes(public_transport_routes_filtered, buffer_m=15)

### Dissolving all the geometry into one polygon
buffer_geom = pt_routes_area_gdf.geometry.iloc[0]

### Intersection each route linestring, and calculating the sum og the linestring inside the buffer
### lines lower than 200m removed
gdf_trips["length_in_buffer_m"] = gdf_trips["linestring"].apply(
    lambda geom: calc_length_within_buffer(geom, buffer_geom, min_length=200)
)

### Adding the gtfs route_length attribut in meters
gdf_trips['route_length'] = gdf_trips.length

### Curvity metric
gdf_trips["curvity"] = gdf_trips["linestring"].apply(calc_curvity)

gdf_trips = add_circular_flag(gdf_trips, threshold=500)


## 3. Passengers attributes

This step loads the validations dataset and creates a shared identifier based on route, direction, alternative, day, and hour.

The same identifier is generated for the trips dataset, allowing a join between trips and passenger data.

As a result, passenger-related attributes (e.g., total passengers and average passengers per bus) are added to each trip record.

In [12]:
### Import the acitvations df
validation = pd.read_csv(r'../data/raw_data/entire_data/grouped_validations.csv')

### Add an idetifier to the activations
validation['route_dir_alt_day_hr'] = (
    validation['RouteId'].fillna(0).astype(int).astype(str) + '_' +
    validation['Direction'].fillna(0).astype(int).astype(str) + '_' +
    validation['Alternative'].astype(str) + '_' +
    validation['DayName'].astype(str) + '_' +
    validation['FullHour'].fillna(0).astype(int).astype(str)
)

### Add the same identifier to the trips gdf
gdf_trips['route_dir_alt_day_hr'] = (
    gdf_trips['route_mkt'].fillna(0).astype(int).astype(str) + '_' +
    gdf_trips['direction'].fillna(0).astype(int).astype(str) + '_' +
    gdf_trips['alternative'].astype(str) + '_' +
    gdf_trips['day'].astype(str) + '_' +
    gdf_trips['hour_rounded'].fillna(0).astype(int).astype(str)
)

### Add the passengers attributes to the trips gdf
gdf_trips = pd.merge(
    gdf_trips,
    validation[['route_dir_alt_day_hr', 'Total_Passengers', 'Avg_Passengers_Per_Bus']],
    on='route_dir_alt_day_hr',
    how='left'
)

C:\Users\shaha\AppData\Local\Temp\ipykernel_41108\3339603526.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  validation = pd.read_csv(r'../data/raw_data/entire_data/grouped_validations.csv')


## 4. Export data

In [13]:
### Export trips withour linestring
gdf_trips.drop(columns = ["linestring"]).to_csv(r'../data/interim_data/telaviv_buses_0704_1304_2024_cleaned_new_features.csv',index=False,encoding='utf-8-sig')

In [14]:
### Export the routes linestring
gdf_trips[['route_id','linestring']].drop_duplicates(subset=['route_id','linestring']).to_csv(r'../data/interim_data/routes_linestrings.csv',index=False,encoding='utf-8-sig')